[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [7]:
# ✏️ YOUR IMPLEMENTATION HERE

def accumulated_step(model, optimizer, loss_fn, micro_batches):

    if not micro_batches:
        return 0.0

    optimizer.zero_grad()

    total_loss = 0.0
    n = len(micro_batches)

    # Micro-batch별 Forward / Backward
    for x, y in micro_batches:
        loss = loss_fn(model(x), y) / n

        loss.backward()
        total_loss += loss.item()

    # 누적된 Gradient로 가중치 업데이트
    optimizer.step()

    return total_loss

In [4]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

Loss: 1.0191145613789558


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_accumulation')


🧪 Testing: Gradient Accumulation (Easy)
──────────────────────────────────────────────────
  ✅ [1/3] Matches full batch update (35.3ms)
  ✅ [2/3] Returns loss value (1.3ms)
  ✅ [3/3] Parameters actually update (1.0ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (37.6ms total)
  Progress saved. Run status() to see your dashboard.



In [6]:
import torch
import torch.nn as nn

# Model
model = nn.Linear(1, 1, bias=False)

with torch.no_grad():
    model.weight.fill_(1.0)

# Optimizer & Loss
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

# Two micro-batches
micro_batches = [
    (torch.tensor([[1.0]]), torch.tensor([[0.0]])),
    (torch.tensor([[2.0]]), torch.tensor([[0.0]]))
]

# Gradient Accumulation
avg_loss = accumulated_step(
    model,
    optimizer,
    loss_fn,
    micro_batches
)

print("Average Loss:", avg_loss)
print("Accumulated Gradient:", model.weight.grad.item())
print("Updated Weight:", model.weight.item())

Average Loss: 2.5
Accumulated Gradient: 5.0
Updated Weight: 0.5


In [8]:
import torch
import torch.nn as nn

# 1. Experimental settings
x = torch.ones(8, 1)
y = torch.arange(1, 9, dtype=torch.float32).unsqueeze(1)

micro_batches = [
    (x[i:i+2], y[i:i+2])
    for i in range(0, 8, 2)
]

loss_fn = nn.MSELoss()
lr = 0.1


# 2. Training experiment
def run_experiment(mode):

    model = nn.Linear(1, 1, bias=False)

    with torch.no_grad():
        model.weight.fill_(0.0)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    weight_history = []
    grad_history = []

    # A. Large Batch: 8 samples at once
    if mode == "large":
        optimizer.zero_grad()

        loss = loss_fn(model(x), y)
        loss.backward()

        grad_history.append(model.weight.grad.item())

        optimizer.step()
        weight_history.append(model.weight.item())

    # B. Small Batch: update every 2 samples
    elif mode == "small":
        for xb, yb in micro_batches:
            optimizer.zero_grad()

            loss = loss_fn(model(xb), yb)
            loss.backward()

            grad_history.append(model.weight.grad.item())

            optimizer.step()
            weight_history.append(model.weight.item())

    # C. Gradient Accumulation: 4 micro-batches
    elif mode == "accum":
        optimizer.zero_grad()

        n = len(micro_batches)

        for xb, yb in micro_batches:
            loss = loss_fn(model(xb), yb) / n
            loss.backward()

        grad_history.append(model.weight.grad.item())

        optimizer.step()
        weight_history.append(model.weight.item())

    # Evaluate on all 8 samples
    with torch.no_grad():
        final_loss = loss_fn(model(x), y).item()

    return grad_history, weight_history, final_loss


# 3. Compare results
for mode in ["large", "small", "accum"]:

    grads, weights, final_loss = run_experiment(mode)

    print(f"\n=== {mode.upper()} ===")
    print("Gradients:", [round(g, 4) for g in grads])
    print("Weight Updates:", [round(w, 4) for w in weights])
    print(f"Final Weight: {weights[-1]:.4f}")
    print(f"Final Loss: {final_loss:.4f}")


=== LARGE ===
Gradients: [-9.0]
Weight Updates: [0.9]
Final Weight: 0.9000
Final Loss: 18.2100

=== SMALL ===
Gradients: [-3.0, -6.4, -9.12, -11.296]
Weight Updates: [0.3, 0.94, 1.852, 2.9816]
Final Weight: 2.9816
Final Loss: 7.5555

=== ACCUM ===
Gradients: [-9.0]
Weight Updates: [0.9]
Final Weight: 0.9000
Final Loss: 18.2100
